In [1]:
import pandas as pd

df = pd.read_csv('../data/chatgpt_reviews_clean_balanced.csv')

In [5]:
!pip install scikit-learn

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 5.8 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 5.8 MB/s  0:00:03 eta 0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

X = vectorizer.fit_transform(df['content'].astype(str))

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(X)

In [10]:
import numpy as np

review_index = 0

similarities = similarity_matrix[review_index]

# самые похожие
similar_indices = np.argsort(similarities)[::-1][1:6]

print("Исходный отзыв:\n")
print(df.iloc[review_index]['content'])

print("\nПохожие отзывы:\n")

for idx in similar_indices:
    print('-' * 50)
    print(df.iloc[idx]['content'])

Исходный отзыв:

I would give this 5 star, but one key flaw it knows what I say without me telling it and that's invasion of privacy

Похожие отзывы:

--------------------------------------------------
it's like I have a guardian who knows everything 😁
--------------------------------------------------
its leaks our privacy
--------------------------------------------------
that's infringement of my privacy
--------------------------------------------------
I give you five star becouse you are very intellingent.
--------------------------------------------------
what limit is it telling me


In [13]:
negative_mask = df['score'].isin([1, 2]).to_numpy()
positive_mask = df['score'].isin([4, 5]).to_numpy()

negative_tfidf = X[negative_mask]
positive_tfidf = X[positive_mask]

In [14]:
import numpy as np

feature_names = vectorizer.get_feature_names_out()

negative_mean = negative_tfidf.mean(axis=0).A1

top_negative = np.argsort(negative_mean)[::-1][:20]

[(feature_names[i], negative_mean[i]) for i in top_negative]

[('app', np.float64(0.04303982635083089)),
 ('ai', np.float64(0.024142624044733694)),
 ('wrong', np.float64(0.023698224846967257)),
 ('bad', np.float64(0.021909613989445843)),
 ('chat', np.float64(0.020730038696905134)),
 ('don', np.float64(0.01974590405667975)),
 ('time', np.float64(0.01968078224479915)),
 ('good', np.float64(0.019581210264751646)),
 ('like', np.float64(0.019446701262109303)),
 ('information', np.float64(0.018160589529889696)),
 ('chatgpt', np.float64(0.018063650609025143)),
 ('use', np.float64(0.017068392576989874)),
 ('answer', np.float64(0.01671392541164842)),
 ('just', np.float64(0.016139161878984138)),
 ('worst', np.float64(0.014025990756794874)),
 ('gpt', np.float64(0.01354466116982106)),
 ('answers', np.float64(0.013373919698363802)),
 ('doesn', np.float64(0.01261531653087195)),
 ('better', np.float64(0.012181537745024488)),
 ('want', np.float64(0.012145451458813345))]

In [15]:
positive_mean = positive_tfidf.mean(axis=0).A1

top_positive = np.argsort(positive_mean)[::-1][:20]

[(feature_names[i], positive_mean[i]) for i in top_positive]

[('app', np.float64(0.087052054727497)),
 ('good', np.float64(0.06511572988743951)),
 ('best', np.float64(0.057385980853827456)),
 ('helpful', np.float64(0.04351864490094638)),
 ('love', np.float64(0.03794236564377237)),
 ('useful', np.float64(0.0291174838014874)),
 ('nice', np.float64(0.029001204760925128)),
 ('chatgpt', np.float64(0.028240957317254827)),
 ('like', np.float64(0.023152308615904307)),
 ('ai', np.float64(0.02302575712328709)),
 ('great', np.float64(0.022256276837224774)),
 ('really', np.float64(0.020754916206175953)),
 ('use', np.float64(0.02038429647073855)),
 ('help', np.float64(0.019280198193433094)),
 ('chat', np.float64(0.018561028489265437)),
 ('helps', np.float64(0.01741772622033324)),
 ('amazing', np.float64(0.017347189838819665)),
 ('gpt', np.float64(0.01591801055969989)),
 ('easy', np.float64(0.013709362157591566)),
 ('students', np.float64(0.013085010773735485))]

In [16]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5, random_state=42)

df['cluster'] = kmeans.fit_predict(X)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: invalid value encountered in matmul
  current_pot = closest_dist_sq @ sample_weight


In [18]:
df['cluster'].value_counts()

cluster
2    3362
1     697
4     291
3     204
0      66
Name: count, dtype: int64

In [19]:
import numpy as np

# слова TF-IDF
terms = vectorizer.get_feature_names_out()

# центры кластеров
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

# топ слов
for i in range(5):
    print(f'\nCLUSTER {i}')
    
    top_words = [
        terms[ind]
        for ind in order_centroids[i, :15]
    ]
    
    print(', '.join(top_words))


CLUSTER 0
working, voice, properly, app, text, isn, good, application, update, chatgpt, time, speech, 100, nice, slow

CLUSTER 1
app, good, nice, helpful, love, useful, like, really, amazing, use, students, great, try, help, work

CLUSTER 2
chatgpt, ai, helpful, good, app, like, use, wrong, time, information, answer, just, don, useful, answers

CLUSTER 3
chat, gpt, good, app, best, love, helpful, thank, like, use, using, really, friend, ai, just

CLUSTER 4
best, app, ai, world, friend, love, chatgpt, used, far, apps, seen, good, aap, just, helpful


In [21]:
cluster_id = 0

sample_reviews = df[df['cluster'] == cluster_id]['content'].head(10)

for review in sample_reviews:
    print('-' * 80)
    print(review)

--------------------------------------------------------------------------------
The mobile version isn’t working well, it keeps getting stuck on the login page, and I can’t access my subscribed account.
--------------------------------------------------------------------------------
mic speech to text not working
--------------------------------------------------------------------------------
On my phone it's not working it's just not opening saying something went wrong try again later
--------------------------------------------------------------------------------
prompt execute button not working properly.
--------------------------------------------------------------------------------
nice working platform
--------------------------------------------------------------------------------
my microphone is not working when I tried to ask chat GPT using my microphone it is not working
--------------------------------------------------------------------------------
It always glitches and